# Smoke test: MedGemma-4b-it as an answerer

Second smoke test, and the one that decides whether the adapter abstraction actually
holds. Qwen2.5-VL validated the harness on a familiar architecture; **MedGemma is a
different model family entirely** (`Gemma3ForConditionalGeneration`), with its own
processor conventions. If `answerer_models.py` needs reshaping to fit it, better to find
that out now than after wiring six models into a production runner.

MedGemma also occupies a specific slot in the Pillar 4a argument: it is **medical but not
pathology**. It tests whether general clinical tuning transfers to histopathology, which
sits between the general-purpose models and the pathology-specialized ones on the
specialization gradient.

## Checks (same structure as the Qwen test, deliberately)

1. Loads at bf16 on one GPU, no CPU offload.
2. Describes a real pathology slide plausibly - the cheapest way to detect a broken
   vision path, which still emits fluent text.
3. Answers MCQs from all three datasets, parsed correctly.
4. **The image matters** - blind condition, sub-pillar 3b.
5. **Option order does not** - position-shuffle, sub-pillar 3b.
6. Throughput, so the full run can be costed.

## One thing already observed

On a first 5-item probe MedGemma answered "B" four times, which looks like positional
bias. Over 30 items it is not: predictions spread A 26.7% / B 30.0% / C 16.7% / D 20.0%,
tracking the ground-truth distribution closely. Small samples produce runs that look like
patterns, which is exactly why section 5 uses the full sample and compares against the
truth distribution rather than eyeballing a handful of replies.

In [ ]:
import os
import sys

# Pick the GPU BEFORE torch initializes CUDA. GPUs 0-2 are busy with the Qwen judge's
# Benchmark 5 run; 4 is free (3 is used by the Qwen2.5-VL smoke test).
os.environ["CUDA_VISIBLE_DEVICES"] = "4"

sys.path.insert(0, os.path.abspath(".."))

import collections
import time

import torch

import tasks
from answerer_models import build_mcq_prompt, load_answerer
from metrics import parse_choice, position_bias, score_predictions

MODEL_KEY = "medgemma"
N_PER_DATASET = 30   # 30, not 20: see the note above on small-sample position artifacts
print(torch.__version__, torch.cuda.device_count(), "visible GPU(s)")

## 1. Load, and verify where the weights landed

In [ ]:
t0 = time.time()
model = load_answerer(MODEL_KEY, device="cuda:0")  # logical 0 == physical 4
print(f"loaded in {time.time() - t0:.1f}s")

devices = collections.Counter(str(p.device) for p in model.model.parameters())
offloaded = [d for d in devices if d in ("cpu", "meta", "disk")]
print(f"parameter devices: {dict(devices)}")
print(f"dtypes: {collections.Counter(str(p.dtype) for p in model.model.parameters())}")
print(f"memory: {model.memory_summary()}")
assert not offloaded, f"model offloaded to {offloaded} - it will be unusably slow"
print("OK: fully resident on GPU, no offload")

# At 4B this is the smallest model in the set (~8 GiB vs ~15 GiB for the 7B models), so
# two MedGemma instances would fit on one A6000 if the full run ever needs the headroom.
print(f"\ntier: {model.spec['tier']}  params: {model.spec['params']}")

## 2. Does it see the image?

Free-text description of a real PathOPEN slide. The point is not to grade the answer but
to confirm the model is describing *this* image. Generic text that would fit any slide
means the vision path is broken - which is how the 4-bit InternVL failure presented in
the judge pipeline: fluent, confident, and unrelated to the image.

In [ ]:
pathopen = tasks.load_pathopen_mcq(shuffle_seed=0)
print(f"{len(pathopen)} PathOPEN MCQ tasks")

sample = pathopen[0]
print(f"image: {os.path.basename(sample.image_path)} ({sample.magnification}, {sample.organ})")
description = model.answer(
    sample.image_path,
    "Describe this pathology image in 2-3 sentences: tissue type, staining, and any "
    "notable cellular features.",
    max_new_tokens=120,
)
print("\n--- description ---")
print(description)

In [ ]:
from PIL import Image

# Look at the slide yourself. No assert can tell a plausible description from an accurate
# one, and this is the step where a pathologist's eye is worth more than any check here.
Image.open(sample.image_path).convert("RGB").resize((512, 512))

## 3. MCQ answering across all three datasets

Each dataset stores options differently (PathOPEN in separate columns; PathMMU/PatchVQA
as letter-prefixed lists with 2-16 options), so all three are exercised rather than
assuming one generalises.

In [ ]:
task_sets = {
    "PathOPEN": pathopen[:N_PER_DATASET],
    "PathMMU": tasks.load_pathmmu_mcq(shuffle_seed=0)[:N_PER_DATASET],
    "PatchVQA": tasks.load_patchvqa_mcq(shuffle_seed=0)[:N_PER_DATASET],
}


def run_condition(task_list, blind=False, shuffle_seed=None, show=0):
    """Predictions for one condition. Returns records shaped for `score_predictions`."""
    records, timings = [], []
    for i, task in enumerate(task_list):
        item = task.shuffled(shuffle_seed) if shuffle_seed is not None else task
        prompt = build_mcq_prompt(item.question, item.lettered_options(), blind=blind)
        t = time.time()
        response = model.answer(None if blind else item.image_path, prompt)
        timings.append(time.time() - t)
        index, method = parse_choice(response, item.options)
        records.append({"item_id": item.item_id, "options": item.options,
                        "correct_index": item.correct_index, "predicted_index": index,
                        "parse_method": method, "raw_response": response})
        if i < show:
            mark = "OK " if index == item.correct_index else "MISS"
            print(f"  {mark} got={response[:34]!r} -> {index} ({method}), "
                  f"correct={item.correct_letter}")
    return records, timings


full_results = {}
for name, task_list in task_sets.items():
    print(f"\n=== {name} ({len(task_list)} items) ===")
    records, timings = run_condition(task_list, show=3)
    full_results[name] = records
    stats = score_predictions(records)
    print(f"  accuracy {stats['accuracy']}%  (chance {stats['chance_accuracy']}%)  "
          f"parse_rate {stats['parse_rate']}%  {sum(timings) / len(timings):.2f}s/item")
    print(f"  parse methods: {stats['parse_methods']}")

## 4. Does the image matter? (blind condition, sub-pillar 3b)

Same questions, image withheld. Two very different things produce a small gap:

- **the questions are answerable from text alone** - a real finding about the dataset,
  and what sub-pillar 3b exists to quantify;
- **the image never reached the model** - a harness bug that would invalidate every
  number in Pillars 3b and 4.

Section 2 already established the model can see images, so a small gap here points at the
dataset rather than the harness. That ordering is deliberate.

MedGemma is worth watching here specifically: broad medical tuning gives it strong
clinical text priors without pathology-specific vision training, so a **larger** blind
accuracy than the general-purpose models would be a substantive result, not a bug.

### Measured on 30 PathOPEN items: full 36.7%, blind 36.7% - a 0pp gap

That looks alarming, and it is worth stating exactly why it is **not** a harness bug:

- **The image does reach the model.** Two different slides produce two different
  descriptions, and with no image the model hallucinates "a vibrant, colorful abstract
  painting" - it clearly notices the absence.
- **The predictions do change.** 25/30 identical, 5 flipped between conditions. The
  flips happened to cancel out, leaving accuracy unchanged.

So a 0pp gap at n=30 is noise, not evidence. This is the strongest argument for running
sub-pillar 3b on the **full** 1,428 tasks rather than a sample: at this size the
diagnostic has no power to distinguish "ignores the image" from "got unlucky". When
reading the table below, treat any gap under ~5pp on 30 items as uninformative.

In [ ]:
print(f"{'dataset':10s} {'full':>8s} {'blind':>8s} {'gap':>8s}  interpretation")
blind_results = {}
for name, task_list in task_sets.items():
    blind_records, _ = run_condition(task_list, blind=True)
    blind_results[name] = blind_records
    full_acc = score_predictions(full_results[name])["accuracy"]
    blind_acc = score_predictions(blind_records)["accuracy"]
    gap = round(full_acc - blind_acc, 2)
    note = ("image is being used" if gap > 5 else
            "little image dependence - check section 2 passed")
    print(f"{name:10s} {full_acc:7.1f}% {blind_acc:7.1f}% {gap:7.1f}pp  {note}")

## 5. Does option order matter? (position-shuffle, sub-pillar 3b)

Three shuffles of the same questions. A model reading the options scores about the same
each time; one keying on position swings.

The `predicted` vs `truth` comparison is the real diagnostic - raw accuracy can hide bias
when the ground truth happens to be spread the same way. Compare the two rows, not the
predicted row alone.

In [ ]:
shuffle_accuracies = {}
for name, task_list in task_sets.items():
    accuracies = []
    for seed in (1, 2, 3):
        records, _ = run_condition(task_list, shuffle_seed=seed)
        accuracies.append(score_predictions(records)["accuracy"])
        if seed == 1:
            bias = position_bias(records)
            print(f"{name} seed1 position bias:")
            print(f"    predicted {bias['predicted_pct']}")
            print(f"    truth     {bias['truth_pct']}")
    shuffle_accuracies[name] = accuracies
    spread = max(accuracies) - min(accuracies)
    flag = "  <- large, investigate" if spread > 15 else ""
    print(f"{name}: {accuracies} -> spread {spread:.1f}pp{flag}\n")

## 6. Verdict and cost projection

In [ ]:
_, timings = run_condition(task_sets["PathOPEN"][:10])
per_item = sum(timings) / len(timings)

n_tasks = len(tasks.load_all_mcq(shuffle_seed=0))
# full + blind + 3 shuffles = 5 predictions per task
calls_per_model = n_tasks * 5
hours = calls_per_model * per_item / 3600

print(f"throughput           {per_item:.2f}s/item")
print(f"tasks (3 datasets)   {n_tasks}")
print(f"calls per model      {calls_per_model} (full + blind + 3 shuffles)")
print(f"this model           {hours:.1f}h")

print("\nchecklist before wiring the full run:")
print("  [ ] description in section 2 is about THIS slide, not generic")
print("  [ ] accuracy comfortably above chance on all three datasets")
print("  [ ] parse_rate near 100% (low rate = prompt problem, not knowledge)")
print("  [ ] blind accuracy clearly below full")
print("  [ ] predicted position distribution tracks truth, not piled on one letter")
print("  [ ] shuffle spread small")

In [ ]:
# Free the GPU so the next smoke test can use it.
del model
torch.cuda.empty_cache()
print("released")